In [1]:
import pandas as pd
import numpy as np

In [2]:
bonds = pd.read_csv('bonds.csv', skiprows=1)
events = pd.read_csv('events.csv')

In [3]:
bonds.head() 

,BondID,Coupon,Frequency,MonthsSinceCoupon
0,BOND1,0.05,2,3
1,BOND2,0.04,1,6
2,BOND3,0.06,2,4
3,BOND4,0.03,1,6
4,BOND5,0.07,2,1


In [4]:
events.head()

,EventID,Desk,Trader,BondID,BuySell,Quantity,CleanPrice
0,1,NY,T_NY_2,BOND1,SELL,50,99.37
1,2,NY,T_NY_2,BOND5,SELL,60,106.56
2,3,NY,T_NY_1,BOND3,SELL,40,103.46
3,4,LN,T_LN_1,BOND5,SELL,20,105.77
4,5,HK,T_HK_2,BOND1,BUY,20,99.64


Metrics: 
- Position
- Accured Intested 
- Dirty Price
- Present value 
- Change in present value 



In [ ]:
# Accrued Interest per bond (constant across events)
# AI = 100 * Coupon * MonthsSinceCoupon / 12
bonds['AI'] = 100 * (bonds['Coupon'] / bonds['frequency']) * bonds['MonthsSinceCoupon'] / 12
ai = bonds.set_index('BondID')['AI'].to_dict()

bonds_list = [f'BOND{i}' for i in range(1, 6)]

# Running state
positions    = {b: 0   for b in bonds_list}
dirty_prices = {b: 0.0 for b in bonds_list}

rows = []
for _, evt in events.iterrows():
    bond  = evt['BondID']
    qty   = evt['Quantity'] if evt['BuySell'] == 'BUY' else -evt['Quantity']

    positions[bond]    += qty
    dirty_prices[bond]  = evt['CleanPrice'] + ai[bond]

    row = {'EventID': evt['EventID']}
    for i, b in enumerate(bonds_list, start=1):
        s = f'stock{i}'
        row[f'{s}position']   = positions[b]
        row[f'{s}dirtyprice'] = round(dirty_prices[b], 4)
        row[f'{s}PV']         = round(dirty_prices[b] * positions[b], 4)
    rows.append(row)

df = pd.DataFrame(rows).set_index('EventID')

# P&L = change in PV from previous event (event 0 baseline is all-zero)
for i in range(1, 6):
    pv_col  = f'stock{i}PV'
    pnl_col = f'stock{i}P&L'
    df[pnl_col] = df[pv_col].diff()
    df.loc[df.index[0], pnl_col] = df.loc[df.index[0], pv_col]  # first event: Δ from 0

# Order columns: position → dirtyprice → PV → P&L, repeated per stock
ordered_cols = []
for i in range(1, 6):
    s = f'stock{i}'
    ordered_cols += [f'{s}position', f'{s}dirtyprice', f'{s}PV', f'{s}P&L']

df = df[ordered_cols]
df

,stock1position,stock1dirtyprice,stock1PV,stock1P&L,stock2position,stock2dirtyprice,stock2PV,stock2P&L,stock3position,stock3dirtyprice,stock3PV,stock3P&L,stock4position,stock4dirtyprice,stock4PV,stock4P&L,stock5position,stock5dirtyprice,stock5PV,stock5P&L
EventID,,,,,,,,,,,,,,,,,,,,
1,-50,100.62,-5031.0,-5031.0,0,0.00,0.0,0.0,0,0.00,0.0,0.0,0,0.00,0.0,0.0,0,0.0000,0.0000,0.0000
2,-50,100.62,-5031.0,0.0,0,0.00,0.0,0.0,0,0.00,0.0,0.0,0,0.00,0.0,0.0,-60,107.1433,-6428.6000,-6428.6000
3,-50,100.62,-5031.0,0.0,0,0.00,0.0,0.0,-40,105.46,-4218.4,-4218.4,0,0.00,0.0,0.0,-60,107.1433,-6428.6000,0.0000
4,-50,100.62,-5031.0,0.0,0,0.00,0.0,0.0,-40,105.46,-4218.4,0.0,0,0.00,0.0,0.0,-80,106.3533,-8508.2667,-2079.6667
5,-30,100.89,-3026.7,2004.3,0,0.00,0.0,0.0,-40,105.46,-4218.4,0.0,0,0.00,0.0,0.0,-80,106.3533,-8508.2667,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,-120,100.94,-12112.8,0.0,240,100.75,24180.0,0.0,140,107.19,15006.6,0.0,720,92.97,66938.4,1635.4,-150,104.6533,-15698.0000,0.0000
147,-120,100.94,-12112.8,0.0,240,100.75,24180.0,0.0,140,107.19,15006.6,0.0,680,93.05,63274.0,-3664.4,-150,104.6533,-15698.0000,0.0000
148,-170,101.04,-17176.8,-5064.0,240,100.75,24180.0,0.0,140,107.19,15006.6,0.0,680,93.05,63274.0,0.0,-150,104.6533,-15698.0000,0.0000
